## Exercise 3: OLTP vs. OLAP Use Cases: Practice with Postgres and DuckDB

In [ ]:
import duckdb
import pandas as pd
import sys
import os
from IPython.core.interactiveshell import InteractiveShell

# Add parent directory to Python path to access utils module
sys.path.insert(0, os.path.abspath('..'))

# Import our utility functions
from utils.db_utils import create_pg_connection, connect_to_duckdb, ensure_tpcc_schema, execute_query
from utils.tpch_utils import transfer_all_tables
from utils.tpcc_utils import (
    create_tpcc_schema, 
    create_tpcc_indexes,
    generate_mock_data, 
    load_data_postgres, 
    load_data_duckdb,
    run_benchmark,
    run_concurrent_benchmark,
    PG_SQL, 
    DUCK_SQL
)

# Configure display settings to prevent output truncation
%config SqlMagic.autopandas = True
%config SqlMagic.feedback = False
%config SqlMagic.displaycon = False
%config InteractiveShell.ast_node_interactivity = "all"

# Stop truncating rows (show all)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

# Clean up databases to start fresh
print("Cleaning up databases...")

# 1. Drop PostgreSQL schemas (tpcc and tables from tpch)
try:
    pg_conn = create_pg_connection()
    with pg_conn.cursor() as cursor:
        cursor.execute("DROP SCHEMA IF EXISTS tpcc CASCADE;")
        # Drop TPC-H tables in public schema
        cursor.execute("""DROP SCHEMA IF EXISTS tpch CASCADE;""")
    pg_conn.commit()
    pg_conn.close()
    print("✓ PostgreSQL schemas and tables dropped")
except Exception as e:
    print(f"PostgreSQL cleanup failed: {e}")

# 2. Clean up DuckDB files
import pathlib
for db_file in ['tpcc.duckdb', 'tpch.duckdb']:
    db_path = pathlib.Path(db_file)
    if db_path.exists():
        db_path.unlink()
        print(f"✓ {db_file} removed")

print("Database cleanup complete - starting fresh!\n")

Cleaning up databases...


In [11]:
%load_ext sql
conn = duckdb.connect(database='tpch.duckdb')
%sql conn --alias duckdb

In [12]:
# --- Configuration ---
SCALE_FACTOR = 2  # Number of warehouses

def main():
    """Main data loading function."""
    print("--- Starting TPC-C Data Load ---")
    
    # 1. Generate Data
    dataframes = generate_mock_data(SCALE_FACTOR)
    
    # 2. Load into PostgreSQL
    pg_conn = None
    try:
        pg_conn = create_pg_connection()
        ensure_tpcc_schema(pg_conn)
        create_tpcc_schema(pg_conn, is_postgres=True)
        load_data_postgres(pg_conn, dataframes)
        # Add indexes to improve OLTP performance
        create_tpcc_indexes(pg_conn)
    except Exception as e:
        print(f"PostgreSQL loading failed: {e}")
    finally:
        if pg_conn:
            pg_conn.close()
            print("PostgreSQL connection closed.")

    # 3. Load into DuckDB
    duck_conn = None
    try:
        duck_conn = connect_to_duckdb(database='tpcc.duckdb')
        create_tpcc_schema(duck_conn, is_postgres=False)
        load_data_duckdb(duck_conn, dataframes)
    except Exception as e:
        print(f"DuckDB loading failed: {e}")
    finally:
        if duck_conn:
            duck_conn.close()
            print("DuckDB connection closed.")
            
    print("\n--- TPC-C Data Load Complete ---")
    print(f"Data scale: {SCALE_FACTOR} warehouse(s)")
    print(f"DuckDB file: tpcc.duckdb")
    print(f"PostgreSQL schema: tpcc")

if __name__ == "__main__":
    main()

--- Starting TPC-C Data Load ---
Generating mock data for 2 warehouse(s)...
Mock data generated.
Successfully ensured 'tpcc' schema exists.
Creating 9 TPC-C tables (PostgreSQL)...
Tables created.
Loading data into PostgreSQL...
  Loading warehouse (2 rows)...
  Loading district (20 rows)...
  Loading customer (60000 rows)...
  Skipping empty table: history
  Loading item (100000 rows)...
  Loading stock (200000 rows)...
  Skipping empty table: orders
  Skipping empty table: new_order
  Skipping empty table: order_line
PostgreSQL load complete in 6.16 seconds.
Creating indexes on PostgreSQL tables...
Indexes created.
PostgreSQL connection closed.
Successfully connected to DuckDB at 'tpcc.duckdb'.
Creating 9 TPC-C tables (DuckDB)...
Tables created.
Loading data into DuckDB...
  Loading warehouse (2 rows)...
  Loading district (20 rows)...
  Loading customer (60000 rows)...
  Skipping empty table: history
  Loading item (100000 rows)...
  Loading stock (200000 rows)...
  Skipping empty ta

In [13]:
# --- Configuration ---
TOTAL_TRANSACTIONS = 400
NUM_WORKERS = 8  # Number of concurrent connections
# ---------------------

def main():
    """Main concurrent benchmark execution."""
    print("--- Starting TPC-C Transaction Test ---")
    print(f"PostgreSQL will use {NUM_WORKERS} concurrent connections to simulate multiple users. DuckDB will run single-threaded (it doesn't support concurrent writes from multiple processes).")
    
    # Connection factory function for PostgreSQL workers
    def create_pg_conn():
        conn = create_pg_connection()
        with conn.cursor() as cursor:
            cursor.execute("SET search_path TO tpcc, public;")
        return conn
    
    # 1. Run PostgreSQL with concurrency
    try:
        run_concurrent_benchmark(
            "PostgreSQL", 
            create_pg_conn, 
            PG_SQL, 
            is_postgres=True, 
            scale_factor=SCALE_FACTOR, 
            total_transactions=TOTAL_TRANSACTIONS,
            num_workers=NUM_WORKERS
        )
    except Exception as e:
        print(f"PostgreSQL concurrent benchmark failed: {e}")
    
    # 2. Run DuckDB single-threaded
    duck_conn = None
    try:
        duck_conn = connect_to_duckdb(database='tpcc.duckdb')
        run_benchmark("DuckDB (single-threaded)", duck_conn, DUCK_SQL, is_postgres=False, 
                     scale_factor=SCALE_FACTOR, total_transactions=TOTAL_TRANSACTIONS)
    except Exception as e:
        print(f"DuckDB benchmark failed: {e}")
    finally:
        if duck_conn:
            duck_conn.close()
            print("\nDuckDB connection closed.")
    
    print("\n--- TPC-C Test Complete ---")
    print("\nKey Takeaway: PostgreSQL handles concurrent transactions efficiently,")
    print("while DuckDB is limited to single-writer scenarios - this is PostgreSQL's OLTP advantage!")

if __name__ == "__main__":
    main()

--- Starting TPC-C Transaction Test ---
PostgreSQL will use 8 concurrent connections to simulate multiple users. DuckDB will run single-threaded (it doesn't support concurrent writes from multiple processes).

--- Running Concurrent TPC-C Benchmark on PostgreSQL ---
  Workers: 8
  Total Transactions: 400
--- PostgreSQL Concurrent Results ---
  Total Transactions: 400
  Total Time: 3.45 seconds
  Transactions Per Second (TPS): 116.06
  Transaction Mix (Success/Total):
    New Order:    185 / 185
    Payment:      177 / 177
    Order Status: 38 / 38
Successfully connected to DuckDB at 'tpcc.duckdb'.

--- Running TPC-C Benchmark on DuckDB (single-threaded) ---
--- DuckDB (single-threaded) Results ---
  Total Transactions: 400
  Total Time: 11.30 seconds
  Transactions Per Second (TPS): 35.39
  Transaction Mix (Success/Total):
    New Order:    183 / 183
    Payment:      173 / 173
    Order Status: 44 / 44

DuckDB connection closed.

--- TPC-C Test Complete ---

Key Takeaway: PostgreSQL h

In [14]:
# Check what was loaded into the TPC-C database
pg_conn = create_pg_connection()
cursor = pg_conn.cursor()

# Set the search path to use the tpcc schema
cursor.execute("SET search_path TO tpcc, public;")

print("=== TPC-C Data Summary ===\n")

# Count records in each table
tables = ["warehouse", "district", "customer", "item", "stock", "orders", "new_order", "order_line", "history"]
for table in tables:
    cursor.execute(f"SELECT COUNT(*) FROM {table};")
    count = cursor.fetchone()[0]
    print(f"{table.upper()}: {count:,} records")

print("\n=== Sample Warehouse Data ===")
cursor.execute("SELECT * FROM warehouse LIMIT 3;")
for row in cursor.fetchall():
    print(row)

print("\n=== Sample District Data ===")
cursor.execute("SELECT d_id, d_w_id, d_name, d_next_o_id FROM district LIMIT 5;")
for row in cursor.fetchall():
    print(row)

print("\n=== Sample Customer Data ===")
cursor.execute("SELECT c_id, c_d_id, c_w_id, c_first, c_last, c_balance FROM customer LIMIT 5;")
for row in cursor.fetchall():
    print(row)

print("\n=== Sample Item Data ===")
cursor.execute("SELECT i_id, i_name, i_price FROM item LIMIT 5;")
for row in cursor.fetchall():
    print(row)

cursor.close()
pg_conn.close()


=== TPC-C Data Summary ===

WAREHOUSE: 2 records
DISTRICT: 20 records
CUSTOMER: 60,000 records
ITEM: 100,000 records
STOCK: 200,000 records
ORDERS: 2,167 records
NEW_ORDER: 2,167 records
ORDER_LINE: 21,645 records
HISTORY: 2,240 records

=== Sample Warehouse Data ===
(2, 'Vasquez-Smith', '53644 Cabrera Dale', 'Apt. 827', 'Kevinville', 'CT', '40822    ', Decimal('0.0490'), Decimal('490715.53'))
(1, 'Salas-Fletcher', '303 Andrew Mountains Apt. 942', 'Suite 559', 'Heidistad', 'NY', '66512    ', Decimal('0.1295'), Decimal('531971.74'))

=== Sample District Data ===
(2, 2, 'Haleyfurt', 3012)
(10, 1, 'Lake James', 3010)
(3, 2, 'East Dustinport', 3009)
(9, 1, 'Port Jillian', 3013)
(7, 1, 'New Sarahhaven', 3012)

=== Sample Customer Data ===
(1, 1, 1, 'Joseph', 'Walker', Decimal('-10.00'))
(2, 1, 1, 'Christopher', 'Cobb', Decimal('-10.00'))
(3, 1, 1, 'Jennifer', 'Hill', Decimal('-10.00'))
(4, 1, 1, 'Crystal', 'Walker', Decimal('-10.00'))
(5, 1, 1, 'Nicholas', 'Simmons', Decimal('-10.00'))

===

In [15]:
# Check what was loaded into DuckDB TPC-C database
duck_conn = connect_to_duckdb(database='tpcc.duckdb', read_only=True)

print("=== DuckDB TPC-C Data Summary ===\n")

# Count records in each table
tables = ["warehouse", "district", "customer", "item", "stock", "orders", "new_order", "order_line", "history"]
for table in tables:
    result = duck_conn.execute(f"SELECT COUNT(*) FROM {table};").fetchone()
    count = result[0]
    print(f"{table.upper()}: {count:,} records")

print("\n=== Sample Warehouse Data ===")
result = duck_conn.execute("SELECT * FROM warehouse LIMIT 3;").fetchall()
for row in result:
    print(row)

print("\n=== Sample District Data ===")
result = duck_conn.execute("SELECT d_id, d_w_id, d_name, d_next_o_id FROM district LIMIT 5;").fetchall()
for row in result:
    print(row)

print("\n=== Sample Customer Data ===")
result = duck_conn.execute("SELECT c_id, c_d_id, c_w_id, c_first, c_last, c_balance FROM customer LIMIT 5;").fetchall()
for row in result:
    print(row)

print("\n=== Sample Item Data ===")
result = duck_conn.execute("SELECT i_id, i_name, i_price FROM item LIMIT 5;").fetchall()
for row in result:
    print(row)

duck_conn.close()

Successfully connected to DuckDB at 'tpcc.duckdb'.
=== DuckDB TPC-C Data Summary ===

WAREHOUSE: 2 records
DISTRICT: 20 records
CUSTOMER: 60,000 records
ITEM: 100,000 records
STOCK: 200,000 records
ORDERS: 183 records
NEW_ORDER: 183 records
ORDER_LINE: 1,825 records
HISTORY: 173 records

=== Sample Warehouse Data ===
(1, 'Salas-Fletcher', '303 Andrew Mountains Apt. 942', 'Suite 559', 'Heidistad', 'NY', '66512', 0.1295, 541907.7299999995)
(2, 'Vasquez-Smith', '53644 Cabrera Dale', 'Apt. 827', 'Kevinville', 'CT', '40822', 0.049, 519826.4500000002)

=== Sample District Data ===
(1, 1, 'Porterside', 3008)
(2, 1, 'Patriciamouth', 3014)
(3, 1, 'North Monicahaven', 3006)
(4, 1, 'West Jason', 3012)
(5, 1, 'New Jeremyport', 3012)

=== Sample Customer Data ===
(1, 1, 1, 'Joseph', 'Walker', -10.0)
(2, 1, 1, 'Christopher', 'Cobb', -10.0)
(3, 1, 1, 'Jennifer', 'Hill', -10.0)
(4, 1, 1, 'Crystal', 'Walker', -10.0)
(5, 1, 1, 'Nicholas', 'Simmons', -10.0)

=== Sample Item Data ===
(1, 'Wife conference 

In [16]:
%%sql
INSTALL tpch;
LOAD tpch;

,Success


In [17]:
%%sql
CALL dbgen(sf = 0.001);

,Success


### Task 1: Select 'query' from tpch_queries table. What does it contain?

In [18]:
%%sql
SELECT 
    query
FROM tpch_queries();

,query
0,"SELECT\n l_returnflag,\n l_linestatus,\n sum(l_quantity) AS sum_qty,\n sum(l_extendedprice) AS sum_base_price,\n sum(l_extendedprice * (1 - l_discount)) AS sum_disc_price,\n sum(l_extendedprice * (1 - l_discount) * (1 + l_tax)) AS sum_charge,\n avg(l_quantity) AS avg_qty,\n avg(l_extendedprice) AS avg_price,\n avg(l_discount) AS avg_disc,\n count(*) AS count_order\nFROM\n lineitem\nWHERE\n l_shipdate <= CAST('1998-09-02' AS date)\nGROUP BY\n l_returnflag,\n l_linestatus\nORDER BY\n l_returnflag,\n l_linestatus;\n"
1,"SELECT\n s_acctbal,\n s_name,\n n_name,\n p_partkey,\n p_mfgr,\n s_address,\n s_phone,\n s_comment\nFROM\n part,\n supplier,\n partsupp,\n nation,\n region\nWHERE\n p_partkey = ps_partkey\n AND s_suppkey = ps_suppkey\n AND p_size = 15\n AND p_type LIKE '%BRASS'\n AND s_nationkey = n_nationkey\n AND n_regionkey = r_regionkey\n AND r_name = 'EUROPE'\n AND ps_supplycost = (\n SELECT\n min(ps_supplycost)\n FROM\n partsupp,\n supplier,\n nation,\n region\n WHERE\n p_partkey = ps_partkey\n AND s_suppkey = ps_suppkey\n AND s_nationkey = n_nationkey\n AND n_regionkey = r_regionkey\n AND r_name = 'EUROPE')\nORDER BY\n s_acctbal DESC,\n n_name,\n s_name,\n p_partkey\nLIMIT 100;\n"
2,"SELECT\n l_orderkey,\n sum(l_extendedprice * (1 - l_discount)) AS revenue,\n o_orderdate,\n o_shippriority\nFROM\n customer,\n orders,\n lineitem\nWHERE\n c_mktsegment = 'BUILDING'\n AND c_custkey = o_custkey\n AND l_orderkey = o_orderkey\n AND o_orderdate < CAST('1995-03-15' AS date)\n AND l_shipdate > CAST('1995-03-15' AS date)\nGROUP BY\n l_orderkey,\n o_orderdate,\n o_shippriority\nORDER BY\n revenue DESC,\n o_orderdate\nLIMIT 10;\n"
3,"SELECT\n o_orderpriority,\n count(*) AS order_count\nFROM\n orders\nWHERE\n o_orderdate >= CAST('1993-07-01' AS date)\n AND o_orderdate < CAST('1993-10-01' AS date)\n AND EXISTS (\n SELECT\n *\n FROM\n lineitem\n WHERE\n l_orderkey = o_orderkey\n AND l_commitdate < l_receiptdate)\nGROUP BY\n o_orderpriority\nORDER BY\n o_orderpriority;\n"
4,"SELECT\n n_name,\n sum(l_extendedprice * (1 - l_discount)) AS revenue\nFROM\n customer,\n orders,\n lineitem,\n supplier,\n nation,\n region\nWHERE\n c_custkey = o_custkey\n AND l_orderkey = o_orderkey\n AND l_suppkey = s_suppkey\n AND c_nationkey = s_nationkey\n AND s_nationkey = n_nationkey\n AND n_regionkey = r_regionkey\n AND r_name = 'ASIA'\n AND o_orderdate >= CAST('1994-01-01' AS date)\n AND o_orderdate < CAST('1995-01-01' AS date)\nGROUP BY\n n_name\nORDER BY\n revenue DESC;\n"
5,SELECT\n sum(l_extendedprice * l_discount) AS revenue\nFROM\n lineitem\nWHERE\n l_shipdate >= CAST('1994-01-01' AS date)\n AND l_shipdate < CAST('1995-01-01' AS date)\n AND l_discount BETWEEN 0.05\n AND 0.07\n AND l_quantity < 24;\n
6,"SELECT\n supp_nation,\n cust_nation,\n l_year,\n sum(volume) AS revenue\nFROM (\n SELECT\n n1.n_name AS supp_nation,\n n2.n_name AS cust_nation,\n extract(year FROM l_shipdate) AS l_year,\n l_extendedprice * (1 - l_discount) AS volume\n FROM\n supplier,\n lineitem,\n orders,\n customer,\n nation n1,\n nation n2\n WHERE\n s_suppkey = l_suppkey\n AND o_orderkey = l_orderkey\n AND c_custkey = o_custkey\n AND s_nationkey = n1.n_nationkey\n AND c_nationkey = n2.n_nationkey\n AND ((n1.n_name = 'FRANCE'\n AND n2.n_name = 'GERMANY')\n OR (n1.n_name = 'GERMANY'\n AND n2.n_name = 'FRANCE'))\n AND l_shipdate BETWEEN CAST('1995-01-01' AS date)\n AND CAST('1996-12-31' AS date)) AS shipping\nGROUP BY\n supp_nation,\n cust_nation,\n l_year\nORDER BY\n supp_nation,\n cust_nation,\n l_year;\n"
7,"SELECT\n o_year,\n sum(\n CASE WHEN nation = 'BRAZIL' THEN\n volume\n ELSE\n 0\n END) / sum(volume) AS mkt_share\nFROM (\n SELECT\n extract(year FROM o_orderdate) AS o_year,\n l_extendedprice * (1 - l_discount) AS volume,\n n2.n_name AS nation\n FROM\n part,\n supplier,\n lineitem,\n orders,\n customer,\n nation n1,\n nation n2,\n region\n WHERE\n p_partkey = l_partkey\n AND s_suppkey = l_suppkey\n AND l_orderkey = o_orderkey\n AND o_custkey = c_custkey\n AND c_nationkey

In [19]:
%%timeit
%%sql
PRAGMA tpch(1);
PRAGMA tpch(2);
PRAGMA tpch(3);
PRAGMA tpch(4);
PRAGMA tpch(5);
PRAGMA tpch(6);
PRAGMA tpch(7);
PRAGMA tpch(8);
PRAGMA tpch(9);
PRAGMA tpch(10);
PRAGMA tpch(11);
PRAGMA tpch(12);
PRAGMA tpch(13);
PRAGMA tpch(14);
PRAGMA tpch(15);
PRAGMA tpch(16);
PRAGMA tpch(17);
PRAGMA tpch(18);
PRAGMA tpch(19);
PRAGMA tpch(20);
PRAGMA tpch(21);

89.7 ms ± 1.22 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [ ]:
# Main execution
# Close any existing DuckDB connection first
try:
    if 'conn' in globals():
        conn.close()
except:
    pass

pg_conn = create_pg_connection()
duck_conn = connect_to_duckdb(database='tpch.duckdb', read_only=False)

transfer_all_tables(duck_conn, pg_conn)

Successfully connected to DuckDB at 'tpch.duckdb'.
Starting TPC-H data transfer from DuckDB to PostgreSQL (schema 'tpch')...
Successfully ensured 'tpch' schema exists.

--- Processing table: region ---
  Extracting 'region' from DuckDB...
  Extracted 5 rows in 0.00 seconds.
  Generating schema for 'region'...
  Loading 'region' into PostgreSQL schema 'tpch' using COPY...


#### Task 2: Import TPCH_QUERY_STRING from the utils library we've prepared.


In [8]:
from utils.tpch_utils import TPCH_QUERY_STRING

ModuleNotFoundError: No module named 'utils'

In [9]:
%%timeit -n 3
execute_query(conn=create_pg_connection(), sql_query=TPCH_QUERY_STRING, fetch=True)

NameError: name 'execute_query' is not defined

You can see that DuckDB is much faster than Postgres on the same type of query for analytical reads.

#### Task 3: Why is DuckDB so much faster than Postgres for analytical reads that involve large aggregations over many elements, and Postgres is better at fetching/writing individual rows?

This is freeform content